# Stack Overflow Developer Technology Analysis

This analysis explores the technologies developers report using and the technologies they would like to work with next.

The notebook covers programming languages, databases, cloud platforms, and web frameworks. It also examines missing values, duplicate records, and category consistency in the fields used for the analysis.

The dataset used here contains 18,845 records and 114 columns.

Technology questions allow multiple selections, so the counts represent technology selections rather than the number of unique respondents.



## Setup: Paths

Run this cell first, before anything else. It works out where the repo root is regardless of whether you launched Jupyter from the repo root or from inside `notebooks/`, so `data/` and `exports/` always resolve to the same two folders next to `README.md`, not wherever the notebook happens to be opened from.

It also looks in a few likely spots for `survey_data_updated.csv` (repo root, `data/`, or right next to the notebook), since that file isn't committed to the repo due to its size.

In [ ]:
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    """Find the project root by looking for README.md and notebooks/."""
    current = start.resolve()

    for candidate in [current] + list(current.parents):
        if (candidate / "README.md").exists() and (candidate / "notebooks").exists():
            return candidate

    return start.resolve()


BASE_DIR = find_repo_root(Path.cwd())

DATA_DIR = BASE_DIR / "data"
EXPORTS_DIR = BASE_DIR / "exports"

DATA_DIR.mkdir(parents=True, exist_ok=True)
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root found: {BASE_DIR.name}/")
print(f"Data dir:        {DATA_DIR.relative_to(BASE_DIR)}/")
print(f"Exports dir:     {EXPORTS_DIR.relative_to(BASE_DIR)}/")

Repo root found: ibm-data-analyst-capstone/
Data dir:        data/
Exports dir:     exports/


###  Imports and Setup

In [ ]:
import requests
import pandas as pd
import json
from openpyxl import Workbook
from bs4 import BeautifulSoup
import requests
import csv

print("Libraries loaded.")

Libraries loaded.


---

##  — Technology Analysis

**Presentation slides:** Programming Language Trends, Database Trends, Dashboard Tabs 1 & 2 

This section loads the Stack Overflow Developer Survey data and generates the 8 top-10 CSV files used as inputs for the Analytics dashboards.

**Output files:**

| File | Dashboard Tab | Chart |
|------|--------------|-------|
| `tab_1_csv_language_top10.csv` | Current Technology Usage | Bar chart |
| `tab_1_csv_database_top10.csv` | Current Technology Usage | Column chart |
| `tab_1_csv_platform_top10.csv` | Current Technology Usage | Word cloud |
| `tab_1_csv_webframe_top10.csv` | Current Technology Usage | Bubble chart |
| `tab_2_csv_language_want_top10.csv` | Future Technology Trend | Bar chart |
| `tab_2_csv_database_want_top10.csv` | Future Technology Trend | Column chart |
| `tab_2_csv_platform_want_top10.csv` | Future Technology Trend | Tree map |
| `tab_2_csv_webframe_want_top10.csv` | Future Technology Trend | Bubble chart |

### 1- Load Survey Data

In [ ]:
survey_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/T3iZyjwN9ifjS-B0JaYVgw/survey-data-updated%205.csv"

df = pd.read_csv(survey_url)

print("Dataset loaded successfully.")
print(f"Dataset shape: {df.shape}")
print(f"Total respondents: {len(df):,}")

df.head(2)

Loaded from: data\survey_data_updated.csv
Dataset shape: (18845, 114)
Total respondents: 18,845


,ResponseId,MainBranch,Age,Employment,RemoteWork,Check,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,...,JobSatPoints_6,JobSatPoints_7,JobSatPoints_8,JobSatPoints_9,JobSatPoints_10,JobSatPoints_11,SurveyLength,SurveyEase,ConvertedCompYearly,JobSat
0,2,I am a developer by profession,35-44 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other...,"Bachelor's degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
1,3,I am a developer by profession,45-54 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other...,"Master's degree (M.A., M.S., M.Eng., MBA, etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,NaN,NaN,NaN,NaN,NaN,NaN,Appropriate in length,Easy,NaN,NaN


### 2- Helper Function

Survey responses use semicolons to separate multiple answers (e.g. `"Python;JavaScript;SQL"`). This function splits, counts, and returns the top N.

In [14]:
def get_top_n(df, column, n=10, label=None):
    """
    Splits semicolon-separated survey responses, counts occurrences,
    and returns the top N as a DataFrame.

    Parameters:
        df     : source DataFrame
        column : column with semicolon-separated values
        n      : number of results to return (default 10)
        label  : output column name (defaults to column name)
    """
    if label is None:
        label = column
    counts = (
        df[column]
        .dropna()
        .str.split(";")
        .explode()
        .str.strip()
        .value_counts()
        .head(n)
        .reset_index()
    )
    counts.columns = [label, "Count"]
    return counts

print("Helper function defined.")

Helper function defined.


### 3 — Current Technology Usage

#### Top 10 Languages — Have Worked With

In [15]:
lang_have = get_top_n(df, "LanguageHaveWorkedWith", label="Language")
print(lang_have.to_string(index=False))
lang_have.to_csv(EXPORTS_DIR / "tab_1_csv_language_top10.csv", index=False)
print("\nSaved: tab_1_csv_language_top10.csv")

               Language  Count
             JavaScript  14943
                    SQL  12602
               HTML/CSS  12410
             TypeScript  10709
                 Python   9590
Bash/Shell (all shells)   7244
                     C#   6340
                   Java   5982
                    PHP   4644
             PowerShell   3438

Saved: tab_1_csv_language_top10.csv


#### Top 10 Databases — Have Worked With

In [16]:
db_have = get_top_n(df, "DatabaseHaveWorkedWith", label="Database")
print(db_have.to_string(index=False))
db_have.to_csv(EXPORTS_DIR / "tab_1_csv_database_top10.csv", index=False)
print("\nSaved: tab_1_csv_database_top10.csv")

            Database  Count
          PostgreSQL  11514
               MySQL   8556
              SQLite   7021
             MongoDB   5930
Microsoft SQL Server   5870
               Redis   5814
             MariaDB   3994
       Elasticsearch   3491
            Dynamodb   2268
              Oracle   1907

Saved: tab_1_csv_database_top10.csv


#### Top 10 Platforms — Have Worked With

In [17]:
platform_have = get_top_n(df, "PlatformHaveWorkedWith", label="Platform")
print(platform_have.to_string(index=False))
platform_have.to_csv(EXPORTS_DIR / "tab_1_csv_platform_top10.csv", index=False)
print("\nSaved: tab_1_csv_platform_top10.csv")

                 Platform  Count
Amazon Web Services (AWS)  10871
          Microsoft Azure   6681
             Google Cloud   5537
               Cloudflare   3874
            Digital Ocean   3170
                 Firebase   3025
                   Vercel   2875
                   Heroku   2002
                  Netlify   1662
                  Hetzner   1392

Saved: tab_1_csv_platform_top10.csv


#### Top 10 Web Frameworks — Have Worked With

In [18]:
webframe_have = get_top_n(df, "WebframeHaveWorkedWith", label="Webframe")
print(webframe_have.to_string(index=False))
webframe_have.to_csv(EXPORTS_DIR / "tab_1_csv_webframe_top10.csv", index=False)
print("\nSaved: tab_1_csv_webframe_top10.csv")

    Webframe  Count
     Node.js   9230
       React   8999
      jQuery   4810
     Express   4408
     Next.js   4399
ASP.NET CORE   4319
     Angular   4038
      Vue.js   3642
     ASP.NET   3094
 Spring Boot   2916

Saved: tab_1_csv_webframe_top10.csv


### 4 — Future Technology Trend

#### Top 10 Languages — Want to Work With

In [19]:
lang_want = get_top_n(df, "LanguageWantToWorkWith", label="Language")
print(lang_want.to_string(index=False))
lang_want.to_csv(EXPORTS_DIR / "tab_2_csv_language_want_top10.csv", index=False)
print("\nSaved: tab_2_csv_language_want_top10.csv")

               Language  Count
             JavaScript  11541
                    SQL  10944
             TypeScript  10437
               HTML/CSS  10016
                 Python   8919
                     Go   5661
                   Rust   5597
                     C#   5590
Bash/Shell (all shells)   5582
                   Java   4048

Saved: tab_2_csv_language_want_top10.csv


#### Top 10 Databases — Want to Work With

In [20]:
db_want = get_top_n(df, "DatabaseWantToWorkWith", label="Database")
print(db_want.to_string(index=False))
db_want.to_csv(EXPORTS_DIR / "tab_2_csv_database_want_top10.csv", index=False)
print("\nSaved: tab_2_csv_database_want_top10.csv")

            Database  Count
          PostgreSQL  12193
               Redis   6384
              SQLite   6295
               MySQL   6204
             MongoDB   5618
Microsoft SQL Server   4345
       Elasticsearch   3665
             MariaDB   3078
            Dynamodb   2154
            Supabase   1623

Saved: tab_2_csv_database_want_top10.csv


#### Top 10 Platforms — Want to Work With

In [21]:
platform_want = get_top_n(df, "PlatformWantToWorkWith", label="Platform")
print(platform_want.to_string(index=False))
platform_want.to_csv(EXPORTS_DIR / "tab_2_csv_platform_want_top10.csv", index=False)
print("\nSaved: tab_2_csv_platform_want_top10.csv")

                 Platform  Count
Amazon Web Services (AWS)  10516
          Microsoft Azure   6380
             Google Cloud   5431
               Cloudflare   4219
            Digital Ocean   2908
                   Vercel   2776
                 Firebase   2720
                  Hetzner   1577
                 Supabase   1368
                  Netlify   1267

Saved: tab_2_csv_platform_want_top10.csv


#### Top 10 Web Frameworks — Want to Work With

In [22]:
webframe_want = get_top_n(df, "WebframeWantToWorkWith", label="Webframe")
print(webframe_want.to_string(index=False))
webframe_want.to_csv(EXPORTS_DIR / "tab_2_csv_webframe_want_top10.csv", index=False)
print("\nSaved: tab_2_csv_webframe_want_top10.csv")

    Webframe  Count
       React   8223
     Node.js   8096
     Next.js   4458
ASP.NET CORE   4166
      Vue.js   3903
     Angular   3309
     Express   3216
     FastAPI   2823
 Spring Boot   2674
      Svelte   2559

Saved: tab_2_csv_webframe_want_top10.csv


### 5- Summary

In [ ]:
files = [
    "tab_1_csv_language_top10.csv",
    "tab_1_csv_database_top10.csv",
    "tab_1_csv_platform_top10.csv",
    "tab_1_csv_webframe_top10.csv",
    "tab_2_csv_language_want_top10.csv",
    "tab_2_csv_database_want_top10.csv",
    "tab_2_csv_platform_want_top10.csv",
    "tab_2_csv_webframe_want_top10.csv",
]

print("All exports generated:")
for fname in files:
    path = EXPORTS_DIR / fname
    rows = pd.read_csv(path).shape[0]
    print(f"  {fname} ({rows} rows)")

print("\n Done. Upload the exports/ folder to Cognos Analytics")

All exports generated:
  tab_1_csv_language_top10.csv (10 rows)
  tab_1_csv_database_top10.csv (10 rows)
  tab_1_csv_platform_top10.csv (10 rows)
  tab_1_csv_webframe_top10.csv (10 rows)
  tab_2_csv_language_want_top10.csv (10 rows)
  tab_2_csv_database_want_top10.csv (10 rows)
  tab_2_csv_platform_want_top10.csv (10 rows)
  tab_2_csv_webframe_want_top10.csv (10 rows)

Done. Upload the exports/ folder to Cognos Analytics.


---
## Data Quality Checks

Before interpreting the technology counts, I checked the dataset for missing values, duplicate records, and inconsistent labels in the technology fields used for the analysis.

These checks help establish whether the fields used in the comparisons are suitable for aggregation.



### 1- Load Raw Data and Take Initial Inventory

In [24]:
# Reload fresh in case Part 3 already modified `df` in memory
raw = pd.read_csv(find_survey_file())

print(f"Shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
print(f"\nColumn list:")
for col in raw.columns:
    print(f"  - {col}")

Shape: 18,845 rows x 114 columns

Column list:
  - ResponseId
  - MainBranch
  - Age
  - Employment
  - RemoteWork
  - Check
  - CodingActivities
  - EdLevel
  - LearnCode
  - LearnCodeOnline
  - TechDoc
  - YearsCode
  - YearsCodePro
  - DevType
  - OrgSize
  - PurchaseInfluence
  - BuyNewTool
  - BuildvsBuy
  - TechEndorse
  - Country
  - Currency
  - CompTotal
  - LanguageHaveWorkedWith
  - LanguageWantToWorkWith
  - LanguageAdmired
  - DatabaseHaveWorkedWith
  - DatabaseWantToWorkWith
  - DatabaseAdmired
  - PlatformHaveWorkedWith
  - PlatformWantToWorkWith
  - PlatformAdmired
  - WebframeHaveWorkedWith
  - WebframeWantToWorkWith
  - WebframeAdmired
  - EmbeddedHaveWorkedWith
  - EmbeddedWantToWorkWith
  - EmbeddedAdmired
  - MiscTechHaveWorkedWith
  - MiscTechWantToWorkWith
  - MiscTechAdmired
  - ToolsTechHaveWorkedWith
  - ToolsTechWantToWorkWith
  - ToolsTechAdmired
  - NEWCollabToolsHaveWorkedWith
  - NEWCollabToolsWantToWorkWith
  - NEWCollabToolsAdmired
  - OpSysPersonal use

### 2- Missingness Profile

For each column, what share of responses are missing. This is the first and simplest readiness signal, and it's invisible once the data has been aggregated into top-10 counts.

In [25]:
missing_profile = (
    raw.isnull().sum()
    .to_frame("missing_count")
    .assign(missing_pct=lambda d: (d["missing_count"] / len(raw) * 100).round(1))
    .sort_values("missing_pct", ascending=False)
)

print(missing_profile.to_string())

                                missing_count  missing_pct
AINextMuch less integrated              18492         98.1
AINextLess integrated                   18093         96.0
AINextNo change                         14468         76.8
AINextMuch more integrated              13676         72.6
EmbeddedAdmired                         13064         69.3
EmbeddedWantToWorkWith                  12932         68.6
EmbeddedHaveWorkedWith                  11943         63.4
AIToolNot interested in Using           10067         53.4
AINextMore integrated                    9816         52.1
ConvertedCompYearly                      9295         49.3
AIToolInterested in Using                7656         40.6
Knowledge_9                              7289         38.7
Knowledge_8                              7261         38.5
Frequency_3                              7262         38.5
Knowledge_7                              7250         38.5
Knowledge_6                              7208         38

**Findings**

- The dataset contains 18,845 rows and 114 columns. Of these, 20 columns have no missing values, while 94 contain at least some missing values.
- Missingness is particularly high in some AI-related follow-up fields. For example, `AINextMuch less integrated` is 98.1% missing and `AINextLess integrated` is 96.0% missing.
- `ConvertedCompYearly` is 49.3% missing, while `CompTotal` is 28.5% missing. Compensation-related analysis would therefore need to account for substantial missingness.
- The eight technology fields used in the current-versus-future comparisons have no missing values in this dataset.

Missing values do not automatically indicate errors. Their meaning depends on the question and how the survey collected the response.

### 3- Duplicate Check

In [26]:
dupe_count = raw.duplicated().sum()
print(f"Fully duplicated rows: {dupe_count}")

# Also check for a likely ID column if one exists (common in SO survey exports: ResponseId)
id_cols = [c for c in raw.columns if "id" in c.lower()]
print(f"\nPossible identifier columns: {id_cols}")
if id_cols:
    for c in id_cols:
        dupe_ids = raw[c].duplicated().sum()
        print(f"  Duplicate values in '{c}': {dupe_ids}")

Fully duplicated rows: 0

Possible identifier columns: ['ResponseId']
  Duplicate values in 'ResponseId': 0


**Findings**

No fully duplicated rows were found. The `ResponseId` field also contains 18,845 unique values across the 18,845 records, with no duplicate IDs.

### 4- Category Consistency Check

Free-text and multi-select fields are the most common source of inconsistent categories (e.g. casing differences, near-duplicate labels). This checks the same fields used in Part 3's `get_top_n()` calls, since those are the fields that feed the dashboards.

In [27]:
category_cols = [
    "LanguageHaveWorkedWith", "LanguageWantToWorkWith",
    "DatabaseHaveWorkedWith", "DatabaseWantToWorkWith",
    "PlatformHaveWorkedWith", "PlatformWantToWorkWith",
    "WebframeHaveWorkedWith", "WebframeWantToWorkWith",
]

for col in category_cols:
    if col not in raw.columns:
        print(f"{col}: NOT FOUND in raw data — check column name against source file")
        continue
    values = raw[col].dropna().str.split(";").explode().str.strip()
    unique_vals = sorted(values.unique())
    print(f"\n{col}: {len(unique_vals)} unique values")
    # Flag likely near-duplicates by comparing lowercased versions
    lower_map = {}
    for v in unique_vals:
        lower_map.setdefault(v.lower(), []).append(v)
    near_dupes = {k: v for k, v in lower_map.items() if len(v) > 1}
    if near_dupes:
        print(f"  Possible casing/near-duplicate labels: {near_dupes}")
    else:
        print("  No casing-based near-duplicates found")


LanguageHaveWorkedWith: 49 unique values
  No casing-based near-duplicates found

LanguageWantToWorkWith: 49 unique values
  No casing-based near-duplicates found

DatabaseHaveWorkedWith: 35 unique values
  No casing-based near-duplicates found

DatabaseWantToWorkWith: 35 unique values
  No casing-based near-duplicates found

PlatformHaveWorkedWith: 27 unique values
  No casing-based near-duplicates found

PlatformWantToWorkWith: 27 unique values
  No casing-based near-duplicates found

WebframeHaveWorkedWith: 36 unique values
  No casing-based near-duplicates found

WebframeWantToWorkWith: 36 unique values
  No casing-based near-duplicates found


**Findings**

No casing-based duplicate labels were found in the eight technology fields checked.

This check helps establish that the technology counts are not being split by labels that differ only in capitalization. It does not rule out every possible spelling variation or other category inconsistency.